# T Cell Analysis: BBKNN Integration + Wilcoxon Marker Finding

**Purpose**: Re-integrate T cells by dataset using BBKNN, then identify marker genes

**Workflow**:
1. Load T cell data (post-cNMF)
2. Inspect dataset distribution and filter small datasets
3. Re-run preprocessing with batch-aware HVG
4. BBKNN batch correction by dataset
5. Clustering and visualization
6. Visualize T cell canonical markers
7. Wilcoxon differential expression
8. Export results

**Author**: r2end  
**Date**: 2025-01-05

## Configuration

In [ ]:
# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

# File paths
INPUT_H5AD = "/home/h2048/data/py/1217/cnmf_batch_production_v1_1_1/T_cells/batch_aware/cnmf_analysis_k40_1/T_cells_with_cnmf_k40.h5ad"
OUTPUT_DIR = "/home/h2048/data/R/0107/t_bbknn_filtered/"

# Epithelial filtering
SCANVI_COLUMN = 'scanvi_predictions'  # Column name for scanvi predictions
# Keywords to identify epithelial cells (case-insensitive matching)
EPITHELIAL_KEYWORDS = ['epithelial', 'basal', 'goblet', 'ciliated', 'secretory', 'club','Epithelial cells']

# Dataset filtering
BATCH_KEY = 'dataset'           # Column name for dataset/batch
MIN_CELLS_PER_DATASET = 20      # Filter datasets with < N cells

# Preprocessing
N_TOP_GENES = 4000              # Number of highly variable genes
N_PCS = 50                      # Number of PCs

# BBKNN parameters
BBKNN_NEIGHBORS_WITHIN_BATCH = 5
BBKNN_N_PCS = 50
BBKNN_TRIM = 45

# UMAP parameters
UMAP_MIN_DIST = 0.4
UMAP_SPREAD = 1.0

# Visualization

# Clustering
LEIDEN_RESOLUTION = 2

# Differential expression
MIN_LOGFC = 0.25
MIN_PCT = 0.1
TOP_N_MARKERS = 50

# Visualization
FIGURE_DPI = 300
FIGURE_FORMAT = 'pdf'
UMAP_SIZE = 3

# Performance
N_JOBS = 8

print("✓ Configuration loaded")

## T Cell Canonical Markers

Define key marker genes for T cell subtypes for validation

In [ ]:
# ============================================================================
# T CELL MARKER GENES
# ============================================================================

# Core T cell markers
TCELL_CORE_MARKERS = {
    'Pan_T': ['CD3D', 'CD3E', 'CD3G'],
    'CD4_T': ['CD4', 'CD40LG'],
    'CD8_T': ['CD8A', 'CD8B'],
    'NK': ['GNLY', 'NKG7', 'KLRD1', 'FCGR3A'],
    'Naive': ['CCR7', 'TCF7', 'LEF1', 'SELL', 'IL7R'],
    'Memory': ['GZMK', 'CD69'],
    'Effector': ['GZMB', 'GZMH', 'PRF1', 'IFNG'],
    'Treg': ['FOXP3', 'IL2RA', 'IKZF2', 'TNFRSF4'],
    'Exhausted': ['PDCD1', 'HAVCR2', 'LAG3', 'TIGIT'],
    'Proliferating': ['MKI67', 'TOP2A', 'STMN1'],
    'Th2': ['GATA3', 'IL4', 'IL5', 'IL13'],
    'Th17': ['RORC', 'IL17A', 'IL23R'],
}

# Flat list for quick visualization
TCELL_KEY_MARKERS = [
    'CD3D', 'CD4', 'CD8A',           # Major lineages
    'GNLY', 'NKG7',                  # NK
    'CCR7', 'SELL', 'IL7R',          # Naive
    'GZMK', 'CD69',                  # Memory
    'GZMB', 'PRF1',                  # Effector
    'FOXP3', 'IL2RA',                # Treg
    'PDCD1', 'HAVCR2',               # Exhausted
    'MKI67'                          # Proliferating
]

print("T cell marker genes defined:")
for category, markers in TCELL_CORE_MARKERS.items():
    print(f"  {category}: {', '.join(markers)}")

## Imports & Setup

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================

import scanpy as sc
import scanpy.external as sce
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import sparse
import warnings
warnings.filterwarnings('ignore')

# Scanpy settings
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=FIGURE_DPI, facecolor='white', frameon=False)
sc.settings.n_jobs = N_JOBS

# Create output directories
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
fig_dir = output_dir / "figures"
fig_dir.mkdir(exist_ok=True)

print("="*80)
print("T Cell BBKNN + Marker Analysis")
print("="*80)
print(f"Output directory: {output_dir}")

## Step 1: Load Data & Inspect

In [ ]:
# ============================================================================
# STEP 1: DATA LOADING
# ============================================================================

print("\n" + "="*80)
print("STEP 1: DATA LOADING & INSPECTION")
print("="*80)

print(f"\nLoading: {INPUT_H5AD}")
adata = sc.read_h5ad(INPUT_H5AD)

print(f"\nData dimensions:")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")

# Check metadata
print(f"\nAvailable .obs columns:")
for col in adata.obs.columns:
    n_unique = adata.obs[col].nunique()
    print(f"  - {col}: {n_unique} unique values")

# Check if cNMF results exist
cnmf_columns = [col for col in adata.obs.columns if 'cnmf' in col.lower() or 'usage' in col.lower()]
if len(cnmf_columns) > 0:
    print(f"\n✓ cNMF results detected:")
    for col in cnmf_columns[:5]:  # Show first 5
        print(f"    {col}")
    print(f"  (and {len(cnmf_columns)-5} more...)" if len(cnmf_columns) > 5 else "")
else:
    print(f"\n⚠️  No cNMF results found")

# Check batch key
if BATCH_KEY not in adata.obs.columns:
    raise ValueError(f"Batch key '{BATCH_KEY}' not found!")

# Dataset distribution
print(f"\nDataset distribution ({BATCH_KEY}):")
dataset_counts = adata.obs[BATCH_KEY].value_counts().sort_values(ascending=False)
print(f"  Total datasets: {len(dataset_counts)}")
print(f"\n  Dataset sizes:")
for dataset, count in dataset_counts.items():
    print(f"    {dataset}: {count:,} cells ({count/adata.n_obs*100:.1f}%)")

# Identify small datasets
small_datasets = dataset_counts[dataset_counts < MIN_CELLS_PER_DATASET]
if len(small_datasets) > 0:
    print(f"\n  ⚠️  Small datasets (< {MIN_CELLS_PER_DATASET} cells): {len(small_datasets)}")
    for dataset, count in small_datasets.items():
        print(f"      {dataset}: {count} cells (will be removed)")

In [ ]:
# ============================================================================
# STEP 2: FILTER EPITHELIAL CELLS
# ============================================================================

print("\n" + "="*80)
print("STEP 2: EPITHELIAL CONTAMINATION FILTERING")
print("="*80)

# Check if scanvi_predict column exists
if SCANVI_COLUMN not in adata.obs.columns:
    print(f"\n⚠️  Warning: '{SCANVI_COLUMN}' column not found!")
    print(f"Available columns: {', '.join(adata.obs.columns)}")
    print(f"\nSkipping epithelial filtering...")
else:
    # Show cell type distribution before filtering
    print(f"\nCell type distribution (before filtering):")
    celltypes_before = adata.obs[SCANVI_COLUMN].value_counts()
    for ct, count in celltypes_before.head(20).items():
        print(f"  {ct}: {count:,} cells")
    if len(celltypes_before) > 20:
        print(f"  ... and {len(celltypes_before)-20} more cell types")
    
    # Identify epithelial cells (case-insensitive)
    epithelial_mask = adata.obs[SCANVI_COLUMN].str.lower().apply(
        lambda x: any(keyword in str(x).lower() for keyword in EPITHELIAL_KEYWORDS)
    )
    
    n_epithelial = epithelial_mask.sum()
    pct_epithelial = 100 * n_epithelial / len(adata)
    
    print(f"\nEpithelial contamination detected:")
    print(f"  Keywords used: {', '.join(EPITHELIAL_KEYWORDS)}")
    print(f"  Epithelial cells: {n_epithelial:,} ({pct_epithelial:.2f}%)")
    
    if n_epithelial > 0:
        # Show which epithelial types were found
        epithelial_types = adata.obs.loc[epithelial_mask, SCANVI_COLUMN].value_counts()
        print(f"\nEpithelial subtypes to be removed:")
        for etype, count in epithelial_types.items():
            print(f"  - {etype}: {count:,} cells")
        
        # Filter out epithelial cells
        print(f"\nRemoving epithelial cells...")
        adata = adata[~epithelial_mask].copy()
        
        print(f"\n✓ Filtering complete")
        print(f"  Cells after filtering: {adata.n_obs:,}")
        print(f"  Cells removed: {n_epithelial:,} ({pct_epithelial:.2f}%)")
        
        # Show remaining cell type distribution
        print(f"\nRemaining cell type distribution:")
        celltypes_after = adata.obs[SCANVI_COLUMN].value_counts()
        for ct, count in celltypes_after.head(15).items():
            print(f"  {ct}: {count:,} cells")
        if len(celltypes_after) > 15:
            print(f"  ... and {len(celltypes_after)-15} more cell types")
    else:
        print(f"\n✓ No epithelial contamination detected")

## Step 2: Filter Small Datasets

In [ ]:
# ============================================================================
# STEP 2: FILTER SMALL DATASETS
# ============================================================================

print("\n" + "="*80)
print("STEP 2: FILTER SMALL DATASETS")
print("="*80)

n_cells_before = adata.n_obs

dataset_counts = adata.obs[BATCH_KEY].value_counts()
valid_datasets = dataset_counts[dataset_counts >= MIN_CELLS_PER_DATASET].index

adata = adata[adata.obs[BATCH_KEY].isin(valid_datasets)].copy()

n_cells_after = adata.n_obs
n_cells_removed = n_cells_before - n_cells_after

print(f"\nFiltering results:")
print(f"  Cells before: {n_cells_before:,}")
print(f"  Cells after: {n_cells_after:,}")
print(f"  Cells removed: {n_cells_removed:,} ({n_cells_removed/n_cells_before*100:.1f}%)")
print(f"  Remaining datasets: {adata.obs[BATCH_KEY].nunique()}")

## Step 3: Data Preparation

In [ ]:
# ============================================================================
# STEP 3: DATA STATE CHECK
# ============================================================================

print("\n" + "="*80)
print("STEP 3: DATA STATE VERIFICATION")
print("="*80)

print(f"\nChecking data structure...")

# Check .X
X_min, X_max = adata.X.min(), adata.X.max()
X_mean = adata.X.mean()
print(f"\n.X matrix:")
print(f"  Type: {type(adata.X)}")
print(f"  Range: [{X_min:.4f}, {X_max:.4f}]")
print(f"  Mean: {X_mean:.4f}")

# Check layers
if 'counts' in adata.layers:
    print(f"\n✓ .layers['counts'] exists")
    HAS_COUNTS = True
else:
    print(f"\n⚠️  .layers['counts'] not found")
    HAS_COUNTS = False

# Prepare counts if needed
if not HAS_COUNTS:
    if X_max > 100:
        print(f"  Saving .X to .layers['counts']")
        adata.layers['counts'] = adata.X.copy()
        HAS_COUNTS = True
    else:
        raise ValueError(".X appears normalized but no counts layer found!")

In [ ]:
# ============================================================================
# STEP 7.5: FILTER UNWANTED GENE CATEGORIES
# ============================================================================
# Insert this cell AFTER Step 7 (Visualization) and BEFORE Step 8 (Wilcoxon DE)

print("\n" + "="*80)
print("STEP 7.5: FILTER UNWANTED GENE CATEGORIES")
print("="*80)

print("\nFiltering genes to improve marker quality...")
print("Will remove: MT, ribosomal, histone, pseudogenes, ENSG, unannotated transcripts")

# ===== Configuration =====
REMOVE_MT = True
REMOVE_RIBO = True
REMOVE_HISTONE = True
REMOVE_PSEUDOGENES = True
REMOVE_ENSG = True
REMOVE_UNANNOTATED = True

# ===== Get all gene names from adata.raw =====
if adata.raw is None:
    print("\n⚠️  Warning: adata.raw is None, will filter adata.var instead")
    all_genes = adata.var_names.tolist()
    use_raw = False
else:
    all_genes = adata.raw.var_names.tolist()
    use_raw = True

print(f"\nTotal genes before filtering: {len(all_genes):,}")

# ===== Initialize list of genes to remove =====
genes_to_remove = []

# ===== 1. Mitochondrial genes (MT-) =====
if REMOVE_MT:
    mt_genes = [g for g in all_genes if g.startswith('MT-')]
    genes_to_remove.extend(mt_genes)
    print(f"\n  1. Mitochondrial genes (MT-): {len(mt_genes)} genes")
    if len(mt_genes) > 0:
        print(f"     Examples: {', '.join(mt_genes[:5])}")

# ===== 2. Ribosomal genes (RPS, RPL, MRPS, MRPL) =====
if REMOVE_RIBO:
    import re
    ribo_pattern = re.compile(r'^(RPS|RPL|MRPS|MRPL)')
    ribo_genes = [g for g in all_genes if ribo_pattern.match(g)]
    genes_to_remove.extend(ribo_genes)
    print(f"  2. Ribosomal genes (RPS/RPL/MRPS/MRPL): {len(ribo_genes)} genes")
    if len(ribo_genes) > 0:
        print(f"     Examples: {', '.join(ribo_genes[:5])}")

# ===== 3. Histone genes (H1, H2A, H2B, H3, H4, HIST) =====
if REMOVE_HISTONE:
    histone_pattern = re.compile(r'^(H1|H2A|H2B|H3|H4|HIST)')
    histone_genes = [g for g in all_genes if histone_pattern.match(g)]
    genes_to_remove.extend(histone_genes)
    print(f"  3. Histone genes (H1/H2A/H2B/H3/H4/HIST): {len(histone_genes)} genes")
    if len(histone_genes) > 0:
        print(f"     Examples: {', '.join(histone_genes[:5])}")

# ===== 4. Pseudogenes (e.g., RPS29P1, RPL10P9) =====
if REMOVE_PSEUDOGENES:
    # Pattern: RPS/RPL/MRPS/MRPL + digits + P + digits
    pseudo_pattern = re.compile(r'^(RPS|RPL|MRPS|MRPL)[0-9]+P[0-9]+$')
    pseudo_genes = [g for g in all_genes if pseudo_pattern.match(g)]
    genes_to_remove.extend(pseudo_genes)
    print(f"  4. Pseudogenes (*P*): {len(pseudo_genes)} genes")
    if len(pseudo_genes) > 0:
        print(f"     Examples: {', '.join(pseudo_genes[:5])}")

# ===== 5. ENSG unannotated genes =====
if REMOVE_ENSG:
    ensg_pattern = re.compile(r'^ENSG[0-9]+')
    ensg_genes = [g for g in all_genes if ensg_pattern.match(g)]
    genes_to_remove.extend(ensg_genes)
    print(f"  5. ENSG unannotated genes: {len(ensg_genes)} genes")
    if len(ensg_genes) > 0:
        print(f"     Examples: {', '.join(ensg_genes[:5])}")

# ===== 6. Unannotated transcripts =====
if REMOVE_UNANNOTATED:
    # Pattern includes:
    # - AC/AL/AP/BX/Z followed by digits and dot
    # - RP followed by digits and dash
    # - CTD-/CTB-/CTC-
    # - LINC followed by digits
    # - Ending with -AS + digits (antisense)
    # - Ending with -OT + digits (overlapping transcript)
    # - Starting with LOC + digits
    unannotated_pattern = re.compile(
        r'^(AC|AL|AP|BX|Z)[0-9]+\.|'
        r'^RP[0-9]+-|'
        r'^CTD-|^CTB-|^CTC-|'
        r'^LINC[0-9]+|'
        r'-AS[0-9]+$|'
        r'-OT[0-9]+$|'
        r'^LOC[0-9]+'
    )
    unannotated_genes = [g for g in all_genes if unannotated_pattern.search(g)]
    genes_to_remove.extend(unannotated_genes)
    print(f"  6. Unannotated transcripts (AC/AL/RP/CTD/LINC/LOC/etc): {len(unannotated_genes)} genes")
    if len(unannotated_genes) > 0:
        print(f"     Examples: {', '.join(unannotated_genes[:5])}")

# ===== Remove duplicates =====
genes_to_remove = list(set(genes_to_remove))
print(f"\n{'─'*80}")
print(f"Total unique genes to remove: {len(genes_to_remove):,}")

# ===== Determine genes to keep =====
genes_to_keep = [g for g in all_genes if g not in genes_to_remove]
print(f"Genes to keep: {len(genes_to_keep):,}")
print(f"Percentage retained: {len(genes_to_keep)/len(all_genes)*100:.1f}%")

# ===== Filter adata.raw =====
if use_raw:
    print(f"\nFiltering adata.raw...")
    # IMPORTANT: Must convert adata.raw to AnnData, filter, then reassign
    # Direct slicing of adata.raw does not work
    
    # Step 1: Convert raw to full AnnData object
    raw_adata = adata.raw.to_adata()
    
    # Step 2: Filter genes
    raw_adata_filtered = raw_adata[:, genes_to_keep].copy()
    
    # Step 3: Reassign to adata.raw
    adata.raw = raw_adata_filtered
    
    print(f"  ✓ adata.raw filtered: {adata.raw.n_vars:,} genes remain")
    
    # Clean up temporary objects
    del raw_adata, raw_adata_filtered
    
else:
    print(f"\nFiltering adata.var...")
    adata = adata[:, genes_to_keep].copy()
    print(f"  ✓ adata filtered: {adata.n_vars:,} genes remain")

# ===== Summary of removed gene categories =====
print(f"\n{'─'*80}")
print("Summary of removed gene categories:")
if REMOVE_MT:
    print(f"  ✓ Mitochondrial genes (MT-)")
if REMOVE_RIBO:
    print(f"  ✓ Ribosomal genes (RPS/RPL/MRPS/MRPL)")
if REMOVE_HISTONE:
    print(f"  ✓ Histone genes (H1/H2A/H2B/H3/H4/HIST)")
if REMOVE_PSEUDOGENES:
    print(f"  ✓ Pseudogenes (*P*)")
if REMOVE_ENSG:
    print(f"  ✓ ENSG unannotated genes")
if REMOVE_UNANNOTATED:
    print(f"  ✓ Unannotated transcripts (AC/AL/RP/CTD/LINC/LOC/etc)")

print("\n✓ Gene filtering complete")
print("  → Subsequent differential expression will use filtered gene set")

## Step 4: Preprocessing

In [ ]:
# ============================================================================
# STEP 4: PREPROCESSING
# ============================================================================

print("\n" + "="*80)
print("STEP 4: PREPROCESSING (HVG + PCA)")
print("="*80)

# Normalize and log
print(f"\nNormalization...")
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
print(f"  ✓ Log-normalized")

# HVG selection
print(f"\nHighly variable genes...")
print(f"  Target: {N_TOP_GENES} genes")
print(f"  Batch key: {BATCH_KEY}")

try:
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=N_TOP_GENES,
        batch_key=BATCH_KEY,
        flavor='seurat_v3',
        subset=False
    )
    print(f"  ✓ Batch-aware HVG completed")
except Exception as e:
    print(f"  ⚠️  Batch-aware failed, using non-batch method")
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=N_TOP_GENES,
        flavor='seurat_v3',
        subset=False
    )
    print(f"  ✓ Non-batch-aware HVG completed")

n_hvg = adata.var['highly_variable'].sum()
print(f"  Selected HVGs: {n_hvg}")

# Save full data to .raw
print(f"\nPreserving full gene data...")
adata.raw = adata.copy()
print(f"  ✓ adata.raw saved (all {adata.n_vars} genes)")

# Subset to HVGs
adata = adata[:, adata.var['highly_variable']].copy()
print(f"  ✓ Subset to {adata.n_vars} HVGs")

# PCA
print(f"\nPCA...")
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=N_PCS, svd_solver='arpack')
print(f"  ✓ PCA completed ({N_PCS} components)")

## Step 5: BBKNN Integration

In [ ]:
# ============================================================================
# STEP 5: BBKNN BATCH CORRECTION
# ============================================================================

print("\n" + "="*80)
print("STEP 5: BBKNN BATCH CORRECTION")
print("="*80)

print(f"\nBBKNN parameters:")
print(f"  Batch key: {BATCH_KEY}")
print(f"  Neighbors within batch: {BBKNN_NEIGHBORS_WITHIN_BATCH}")
print(f"  Number of PCs: {BBKNN_N_PCS}")
print(f"  Trim: {BBKNN_TRIM}")  # ← 添加这一行显示trim参数
print(f"  Number of datasets: {adata.obs[BATCH_KEY].nunique()}")

print(f"\nRunning BBKNN...")
sce.pp.bbknn(
    adata,
    batch_key=BATCH_KEY,
    neighbors_within_batch=BBKNN_NEIGHBORS_WITHIN_BATCH,
    n_pcs=BBKNN_N_PCS,
    trim=BBKNN_TRIM  # ← 修改这一行，从trim=None改为trim=BBKNN_TRIM
)
print(f"  ✓ BBKNN completed")

# UMAP
print(f"\nComputing UMAP...")
print(f"  Parameters:")
print(f"    - min_dist: {UMAP_MIN_DIST}")
print(f"    - spread: {UMAP_SPREAD}")
sc.tl.umap(adata, min_dist=UMAP_MIN_DIST, spread=UMAP_SPREAD)
print(f"  ✓ UMAP completed")

## Step 6: Clustering

In [ ]:
# ============================================================================
# STEP 6: LEIDEN CLUSTERING
# ============================================================================

print("\n" + "="*80)
print("STEP 6: LEIDEN CLUSTERING")
print("="*80)

print(f"\nClustering parameters:")
print(f"  Method: Leiden")
print(f"  Resolution: {LEIDEN_RESOLUTION}")

sc.tl.leiden(adata, resolution=LEIDEN_RESOLUTION, key_added='leiden')

n_clusters = adata.obs['leiden'].nunique()
print(f"\n✓ Clustering completed: {n_clusters} clusters")

print(f"\nCluster sizes:")
cluster_counts = adata.obs['leiden'].value_counts().sort_index()
for cluster, count in cluster_counts.items():
    print(f"  Cluster {cluster}: {count:,} cells ({count/adata.n_obs*100:.1f}%)")

## Step 7: Basic Visualization

In [ ]:
# ============================================================================
# STEP 7: BASIC VISUALIZATION
# ============================================================================

print("\n" + "="*80)
print("STEP 7: VISUALIZATION")
print("="*80)

# UMAP by cluster
print(f"\nGenerating UMAP plots...")

fig, ax = plt.subplots(figsize=(8, 6))
sc.pl.umap(
    adata,
    color='leiden',
    ax=ax,
    show=False,
    legend_loc='on data',
    legend_fontsize=8,
    size=UMAP_SIZE,
    title='T Cell Clusters (BBKNN-integrated)'
)
plt.tight_layout()
plt.savefig(fig_dir / f'01_umap_clusters.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f"  ✓ Saved: 01_umap_clusters.{FIGURE_FORMAT}")

# UMAP by dataset
fig, ax = plt.subplots(figsize=(8, 6))
sc.pl.umap(
    adata,
    color=BATCH_KEY,
    ax=ax,
    show=False,
    size=UMAP_SIZE,
    title=f'Batch Mixing ({BATCH_KEY})'
)
plt.tight_layout()
plt.savefig(fig_dir / f'02_umap_batch.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f"  ✓ Saved: 02_umap_batch.{FIGURE_FORMAT}")


# UMAP by dataset
fig, ax = plt.subplots(figsize=(8, 6))
sc.pl.umap(
    adata,
    color=SCANVI_COLUMN,
    ax=ax,
    show=False,
    size=UMAP_SIZE,
    title=f'ScanVI Predictions ({SCANVI_COLUMN})'
)
plt.tight_layout()
plt.savefig(fig_dir / f'03_umap_scanvi.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f"  ✓ Saved: 03_umap_scanvi.{FIGURE_FORMAT}")

# Combined view
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sc.pl.umap(adata, color='leiden', ax=axes[0], show=False, legend_loc='on data', size=UMAP_SIZE)
sc.pl.umap(adata, color=BATCH_KEY, ax=axes[1], show=False, size=UMAP_SIZE)
axes[0].set_title('Clusters', fontsize=14, weight='bold')
axes[1].set_title('Dataset Batch', fontsize=14, weight='bold')
plt.tight_layout()
plt.savefig(fig_dir / f'03_umap_combined.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f"  ✓ Saved: 03_umap_combined.{FIGURE_FORMAT}")

## Step 8: T Cell Marker Visualization

In [ ]:
# ============================================================================
# STEP 8: T CELL MARKER VISUALIZATION
# ============================================================================

print("\n" + "="*80)
print("STEP 8: T CELL CANONICAL MARKER VISUALIZATION")
print("="*80)

# Check which markers are available
available_markers = [m for m in TCELL_KEY_MARKERS if m in adata.raw.var_names]
missing_markers = [m for m in TCELL_KEY_MARKERS if m not in adata.raw.var_names]

print(f"\nMarker availability:")
print(f"  Available: {len(available_markers)}/{len(TCELL_KEY_MARKERS)}")
if len(missing_markers) > 0:
    print(f"  Missing: {', '.join(missing_markers)}")

if len(available_markers) > 0:
    # UMAP grid
    print(f"\nGenerating marker UMAP grid...")
    n_cols = 4
    n_rows = int(np.ceil(len(available_markers) / n_cols))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes]
    
    for idx, marker in enumerate(available_markers):
        sc.pl.umap(
            adata,
            color=marker,
            ax=axes[idx],
            show=False,
            use_raw=True,
            vmax='p99',
            frameon=False,
            size=UMAP_SIZE,
            title=marker
        )
    
    # Hide extra subplots
    for idx in range(len(available_markers), len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(fig_dir / f'04_tcell_markers_umap.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
    plt.show()
    print(f"  ✓ Saved: 04_tcell_markers_umap.{FIGURE_FORMAT}")
    
    # Dotplot
    print(f"\nGenerating marker dotplot...")
    try:
        fig = sc.pl.dotplot(
            adata,
            var_names=available_markers,
            groupby='leiden',
            use_raw=True,
            show=False,
            figsize=(14, 6),
            standard_scale='var'
        )
        plt.savefig(fig_dir / f'05_tcell_markers_dotplot.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
        plt.show()
        print(f"  ✓ Saved: 05_tcell_markers_dotplot.{FIGURE_FORMAT}")
    except Exception as e:
        print(f"  ⚠️  Dotplot failed: {e}")

else:
    print(f"\n⚠️  No canonical T cell markers found in dataset")

## Step 9: Wilcoxon Differential Expression

In [ ]:
# ============================================================================
# STEP 9: WILCOXON DIFFERENTIAL EXPRESSION
# ============================================================================

print("\n" + "="*80)
print("STEP 9: DIFFERENTIAL EXPRESSION ANALYSIS")
print("="*80)

print(f"\nRunning Wilcoxon rank-sum test...")
print(f"  Method: One-vs-rest")
print(f"  Using: adata.raw (all genes)")
print(f"  Top N genes: {TOP_N_MARKERS}")

sc.tl.rank_genes_groups(
    adata,
    groupby='leiden',
    method='wilcoxon',
    use_raw=True,
    n_genes=TOP_N_MARKERS,
    key_added='rank_genes_wilcox'
)
print(f"  ✓ Differential expression completed")

## Step 10: Extract & Filter Markers

In [ ]:
# ============================================================================
# STEP 10: EXTRACT & FILTER MARKERS
# ============================================================================

print("\n" + "="*80)
print("STEP 10: EXTRACTING & FILTERING MARKERS")
print("="*80)

print(f"\nFiltering criteria:")
print(f"  - log2FC > {MIN_LOGFC}")
print(f"  - Adjusted p-value < 0.05")
print(f"  - Expression % > {MIN_PCT*100}%")

all_markers = []

for cluster in adata.obs['leiden'].cat.categories:
    cluster_result = sc.get.rank_genes_groups_df(
        adata,
        group=cluster,
        key='rank_genes_wilcox'
    )
    
    cluster_result_filtered = cluster_result[
        (cluster_result['logfoldchanges'] > MIN_LOGFC) &
        (cluster_result['pvals_adj'] < 0.05)
    ].copy()
    
    if len(cluster_result_filtered) == 0:
        print(f"  Cluster {cluster}: No significant markers")
        continue
    
    # Calculate expression percentage
    cluster_cells = adata.raw.X[adata.obs['leiden'] == cluster]
    other_cells = adata.raw.X[adata.obs['leiden'] != cluster]
    
    pct_in = []
    pct_out = []
    
    for gene in cluster_result_filtered['names']:
        gene_idx = adata.raw.var_names.get_loc(gene)
        
        if sparse.issparse(cluster_cells):
            expr_in = cluster_cells[:, gene_idx].toarray().flatten()
            expr_out = other_cells[:, gene_idx].toarray().flatten()
        else:
            expr_in = cluster_cells[:, gene_idx].flatten()
            expr_out = other_cells[:, gene_idx].flatten()
        
        pct_in.append(np.sum(expr_in > 0) / len(expr_in))
        pct_out.append(np.sum(expr_out > 0) / len(expr_out))
    
    cluster_result_filtered['cluster'] = cluster
    cluster_result_filtered['pct_in_cluster'] = pct_in
    cluster_result_filtered['pct_out_cluster'] = pct_out
    
    cluster_result_filtered = cluster_result_filtered[
        cluster_result_filtered['pct_in_cluster'] > MIN_PCT
    ]
    
    n_markers = len(cluster_result_filtered)
    print(f"  Cluster {cluster}: {n_markers} markers")
    
    all_markers.append(cluster_result_filtered)

# Combine results
if len(all_markers) > 0:
    markers_df = pd.concat(all_markers, ignore_index=True)
    markers_df = markers_df.sort_values(['cluster', 'pvals_adj'])
    
    markers_df.to_csv(output_dir / "tcell_markers_wilcox_all.csv", index=False)
    print(f"\n✓ Total markers: {len(markers_df):,}")
    print(f"✓ Saved: tcell_markers_wilcox_all.csv")
    
    top_markers = markers_df.groupby('cluster').head(10)
    top_markers.to_csv(output_dir / "tcell_markers_wilcox_top10.csv", index=False)
    print(f"✓ Saved: tcell_markers_wilcox_top10.csv")
else:
    print(f"\n⚠️  No markers passed filtering!")
    markers_df = pd.DataFrame()

In [ ]:
# ============================================================================
# STEP 9.5: CD3/CD4/CD8 EXPRESSION & SCANVI PREDICTIONS ANALYSIS
# ============================================================================

print("\n" + "="*80)
print("STEP 9.5: CD3/CD4/CD8 EXPRESSION ANALYSIS")
print("="*80)

# Define key T cell lineage markers
tcell_lineage_markers = ['CD3D', 'CD3E', 'CD3G', 'CD4', 'CD8A', 'CD8B']

# Check which markers are available in raw data
available_markers = [g for g in tcell_lineage_markers if g in adata.raw.var_names]
missing_markers = [g for g in tcell_lineage_markers if g not in adata.raw.var_names]

print(f"\nAvailable markers: {', '.join(available_markers)}")
if missing_markers:
    print(f"Missing markers: {', '.join(missing_markers)}")

# --- Part 1: Per-cell expression data ---
print(f"\nExtracting per-cell expression...")

# Get expression from raw data (log1p normalized)
cell_expr_data = []
for gene in available_markers:
    gene_idx = adata.raw.var_names.get_loc(gene)
    
    if sparse.issparse(adata.raw.X):
        expr = adata.raw.X[:, gene_idx].toarray().flatten()
    else:
        expr = adata.raw.X[:, gene_idx].flatten()
    
    cell_expr_data.append(expr)

# Create per-cell dataframe
cell_expr_df = pd.DataFrame(
    dict(zip(available_markers, cell_expr_data)),
    index=adata.obs_names
)

# Add cluster information
cell_expr_df['leiden_cluster'] = adata.obs['leiden'].values

# Add scanvi_predictions if available
if 'scanvi_predictions' in adata.obs.columns:
    cell_expr_df['scanvi_predictions'] = adata.obs['scanvi_predictions'].values
else:
    print("⚠️  'scanvi_predictions' column not found in adata.obs")

# Save per-cell expression
cell_expr_output = output_dir / "tcell_cd3_cd4_cd8_per_cell.csv"
cell_expr_df.to_csv(cell_expr_output)
print(f"✓ Saved per-cell expression: tcell_cd3_cd4_cd8_per_cell.csv")
print(f"  Shape: {cell_expr_df.shape}")

# --- Part 2: Per-cluster summary statistics ---
print(f"\nCalculating per-cluster statistics...")

cluster_stats = []
for cluster in sorted(adata.obs['leiden'].unique()):
    cluster_mask = adata.obs['leiden'] == cluster
    n_cells = cluster_mask.sum()
    
    cluster_row = {'cluster': cluster, 'n_cells': n_cells}
    
    for gene in available_markers:
        gene_idx = adata.raw.var_names.get_loc(gene)
        
        if sparse.issparse(adata.raw.X):
            expr = adata.raw.X[cluster_mask, gene_idx].toarray().flatten()
        else:
            expr = adata.raw.X[cluster_mask, gene_idx].flatten()
        
        # Calculate statistics
        cluster_row[f'{gene}_mean_expr'] = expr.mean()
        cluster_row[f'{gene}_pct_positive'] = (expr > 0).sum() / len(expr) * 100
    
    cluster_stats.append(cluster_row)

cluster_stats_df = pd.DataFrame(cluster_stats)

# Save cluster statistics
cluster_stats_output = output_dir / "tcell_cd3_cd4_cd8_cluster_stats.csv"
cluster_stats_df.to_csv(cluster_stats_output, index=False)
print(f"✓ Saved cluster statistics: tcell_cd3_cd4_cd8_cluster_stats.csv")

# Display summary
print(f"\nCluster expression summary:")
print(cluster_stats_df.to_string(index=False))

# --- Part 3: scanvi_predictions distribution ---
if 'scanvi_predictions' in adata.obs.columns:
    print("\n" + "="*80)
    print("SCANVI PREDICTIONS DISTRIBUTION")
    print("="*80)
    
    scanvi_counts = adata.obs['scanvi_predictions'].value_counts()
    scanvi_pct = (scanvi_counts / len(adata)) * 100
    
    print(f"\nTotal cell types predicted: {len(scanvi_counts)}")
    print(f"\nTop 20 predicted cell types:")
    print("─" * 80)
    print(f"{'Cell Type':<40} {'Count':>10} {'Percentage':>10}")
    print("─" * 80)
    
    for cell_type, count in scanvi_counts.head(20).items():
        pct = scanvi_pct[cell_type]
        print(f"{cell_type:<40} {count:>10,} {pct:>9.2f}%")
    
    # Save full distribution
    scanvi_dist = pd.DataFrame({
        'cell_type': scanvi_counts.index,
        'count': scanvi_counts.values,
        'percentage': scanvi_pct.values
    })
    scanvi_output = output_dir / "tcell_scanvi_predictions_distribution.csv"
    scanvi_dist.to_csv(scanvi_output, index=False)
    print(f"\n✓ Saved scanvi distribution: tcell_scanvi_predictions_distribution.csv")
    
    # Identify dominant cell types (>5%)
    dominant_types = scanvi_dist[scanvi_dist['percentage'] > 5.0]
    print(f"\nDominant cell types (>5%):")
    for _, row in dominant_types.iterrows():
        print(f"  • {row['cell_type']}: {row['percentage']:.1f}%")
    
    # Cross-tabulation: leiden clusters vs scanvi predictions
    print(f"\nCross-tabulation: Leiden clusters vs scANVI predictions")
    cross_tab = pd.crosstab(
        adata.obs['leiden'],
        adata.obs['scanvi_predictions'],
        normalize='index'
    ) * 100
    
    # Save cross-tabulation
    cross_tab_output = output_dir / "tcell_leiden_vs_scanvi_crosstab.csv"
    cross_tab.to_csv(cross_tab_output)
    print(f"✓ Saved cross-tabulation: tcell_leiden_vs_scanvi_crosstab.csv")
    
    # Show dominant prediction per cluster
    print(f"\nDominant scANVI prediction per Leiden cluster:")
    for cluster in sorted(adata.obs['leiden'].unique()):
        cluster_data = adata.obs[adata.obs['leiden'] == cluster]
        top_pred = cluster_data['scanvi_predictions'].value_counts().head(1)
        if len(top_pred) > 0:
            cell_type = top_pred.index[0]
            count = top_pred.values[0]
            pct = (count / len(cluster_data)) * 100
            print(f"  Cluster {cluster}: {cell_type} ({pct:.1f}%, n={count})")

print("\n✓ CD3/CD4/CD8 analysis and scanvi predictions check complete")

In [ ]:
epi_genes = ["EPCAM","KRT8","KRT18","KRT19"]
epi_genes = [g for g in epi_genes if g in adata.var_names]
cluster_key = 'leiden'
import scanpy as sc
sc.tl.score_genes(adata, gene_list=epi_genes, score_name="epi_score")

# Inspect by cluster
import pandas as pd
tmp = adata.obs[[cluster_key, "epi_score"]].copy()
print(tmp.groupby(cluster_key)["epi_score"].describe()[["mean","50%","75%","max"]].sort_values("mean", ascending=False).head(15))


## Step 11: Marker Visualization

In [ ]:
# ============================================================================
# STEP 11: MARKER VISUALIZATION
# ============================================================================

if len(markers_df) > 0:
    print("\n" + "="*80)
    print("STEP 11: MARKER VISUALIZATION")
    print("="*80)
    
    # Heatmap
    print(f"\nGenerating heatmap (top 5 per cluster)...")
    try:
        fig = sc.pl.rank_genes_groups_heatmap(
            adata,
            n_genes=5,
            key='rank_genes_wilcox',
            use_raw=True,
            show=False,
            cmap='RdBu_r',
            figsize=(12, 10),
            vmin=-3,
            vmax=3,
            dendrogram=False
        )
        plt.savefig(fig_dir / f'06_marker_heatmap.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
        plt.show()
        print(f"  ✓ Saved: 06_marker_heatmap.{FIGURE_FORMAT}")
    except Exception as e:
        print(f"  ⚠️  Heatmap failed: {e}")
    
    # Dotplot
    print(f"\nGenerating dotplot (top 3 per cluster)...")
    try:
        fig = sc.pl.rank_genes_groups_dotplot(
            adata,
            n_genes=3,
            key='rank_genes_wilcox',
            use_raw=True,
            show=False,
            figsize=(14, 6)
        )
        plt.savefig(fig_dir / f'07_marker_dotplot.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
        plt.show()
        print(f"  ✓ Saved: 07_marker_dotplot.{FIGURE_FORMAT}")
    except Exception as e:
        print(f"  ⚠️  Dotplot failed: {e}")

print("\n✓ Visualization complete")

In [ ]:
# ============================================================================
# CALCULATE T CELL MARKER EXPRESSION PER CLUSTER
# ============================================================================

print("\n" + "="*80)
print("CALCULATING T CELL MARKER EXPRESSION PER CLUSTER")
print("="*80)

# Extended marker list (add more markers beyond the core set)
TCELL_EXTENDED_MARKERS = {
    **TCELL_CORE_MARKERS,
    'Tissue_resident': ['ITGAE', 'CXCR6', 'ZNF683', 'CD69'],
    'Central_memory': ['CCR7', 'SELL', 'TCF7'],
    'Effector_memory': ['KLRG1', 'CX3CR1'],
    'Tfh': ['CXCR5', 'BCL6', 'ICOS', 'PDCD1'],
    'Secretory_contam': ['SCGB1A1', 'SCGB3A1', 'MUC5AC'],
    'Myeloid_contam': ['S100A8', 'S100A9', 'LYZ', 'CD14'],
    'Stress': ['HSPA1A', 'HSPA1B', 'DNAJB1', 'HSPD1'],
    'Gamma_delta': ['TRGV9', 'TRDV2', 'KLRC1', 'KLRC2'],
    'MAIT': ['SLC4A10', 'KLRB1', 'NCR3', 'TRAV1-2'],
    'IEG_activated': ['FOS', 'JUN', 'EGR1', 'NR4A1'],
}

# Flatten all markers
all_markers = []
for category, genes in TCELL_EXTENDED_MARKERS.items():
    all_markers.extend(genes)
all_markers = list(set(all_markers))  # Remove duplicates

print(f"\nTotal unique markers to calculate: {len(all_markers)}")

# Filter to available genes
if adata.raw is not None:
    available_genes = adata.raw.var_names
    data_source = "adata.raw"
else:
    available_genes = adata.var_names
    data_source = "adata"

available_markers = [g for g in all_markers if g in available_genes]
missing_markers = set(all_markers) - set(available_markers)

print(f"Available markers: {len(available_markers)}")
print(f"Missing markers: {len(missing_markers)}")
if len(missing_markers) > 0 and len(missing_markers) <= 20:
    print(f"  Missing: {', '.join(sorted(missing_markers))}")

# Calculate expression per cluster
print(f"\nCalculating expression using {data_source}...")

results = []
clusters = sorted(adata.obs['leiden'].unique(), key=lambda x: int(x) if str(x).isdigit() else 999)

for cluster in clusters:
    cluster_mask = adata.obs['leiden'] == cluster
    n_cells = cluster_mask.sum()
    
    # Get data for this cluster
    if adata.raw is not None:
        cluster_data = adata.raw.X[cluster_mask, :]
        var_names = adata.raw.var_names
    else:
        cluster_data = adata.X[cluster_mask, :]
        var_names = adata.var_names
    
    for gene in available_markers:
        gene_idx = var_names.get_loc(gene)
        
        # Extract expression values
        if sparse.issparse(cluster_data):
            expr_vals = cluster_data[:, gene_idx].toarray().flatten()
        else:
            expr_vals = cluster_data[:, gene_idx].flatten()
        
        mean_expr = expr_vals.mean()
        pct_positive = (expr_vals > 0).sum() / len(expr_vals) * 100
        
        results.append({
            'cluster': str(cluster),
            'gene': gene,
            'mean_expr': mean_expr,
            'pct_positive': pct_positive,
            'n_cells': n_cells
        })
    
    if (int(cluster) + 1) % 10 == 0:
        print(f"  Processed {int(cluster) + 1}/{len(clusters)} clusters...")

print(f"\n✓ Calculation complete for {len(clusters)} clusters")

# Convert to DataFrame
df_marker_expr = pd.DataFrame(results)

# Save long format
output_long = output_dir / "tcell_marker_expression_per_cluster_long.csv"
df_marker_expr.to_csv(output_long, index=False)
print(f"\n✓ Saved long format: {output_long.name}")

# Create wide format (mean expression)
df_mean_wide = df_marker_expr.pivot(index='cluster', columns='gene', values='mean_expr')
df_mean_wide.insert(0, 'n_cells', df_marker_expr.groupby('cluster')['n_cells'].first())

output_mean = output_dir / "tcell_marker_mean_expression_wide.csv"
df_mean_wide.to_csv(output_mean)
print(f"✓ Saved mean expression (wide): {output_mean.name}")

# Create wide format (percent positive)
df_pct_wide = df_marker_expr.pivot(index='cluster', columns='gene', values='pct_positive')
df_pct_wide.insert(0, 'n_cells', df_marker_expr.groupby('cluster')['n_cells'].first())

output_pct = output_dir / "tcell_marker_pct_positive_wide.csv"
df_pct_wide.to_csv(output_pct)
print(f"✓ Saved percent positive (wide): {output_pct.name}")

# Create category summary (average across genes in each category)
print(f"\nCreating category summaries...")

category_results = []
for category, genes in TCELL_EXTENDED_MARKERS.items():
    # Filter to available genes in this category
    category_genes = [g for g in genes if g in available_markers]
    
    if len(category_genes) == 0:
        continue
    
    # Get data for this category
    category_data = df_marker_expr[df_marker_expr['gene'].isin(category_genes)]
    
    # Average across genes
    category_summary = category_data.groupby('cluster').agg({
        'mean_expr': 'mean',
        'pct_positive': 'mean',
        'n_cells': 'first'
    }).reset_index()
    
    category_summary.columns = ['cluster', f'{category}_mean', f'{category}_pct', 'n_cells']
    category_results.append(category_summary)

# Merge all categories
if len(category_results) > 0:
    df_category = category_results[0][['cluster', 'n_cells']]
    for cat_df in category_results:
        category_name = cat_df.columns[1].replace('_mean', '')
        df_category = df_category.merge(
            cat_df[['cluster', f'{category_name}_mean', f'{category_name}_pct']],
            on='cluster',
            how='outer'
        )
    
    output_category = output_dir / "tcell_marker_category_summary.csv"
    df_category.to_csv(output_category, index=False)
    print(f"✓ Saved category summary: {output_category.name}")

# Display sample results
print(f"\n{'='*80}")
print("SAMPLE RESULTS (First 3 clusters, key markers)")
print(f"{'='*80}\n")

key_display_markers = ['CD3D', 'CD4', 'CD8A', 'GNLY', 'NKG7', 'CCR7', 'GZMB', 'FOXP3', 'MKI67']
key_display_markers = [m for m in key_display_markers if m in available_markers]

sample_clusters = clusters[:3]
sample_data = df_marker_expr[
    (df_marker_expr['cluster'].isin([str(c) for c in sample_clusters])) &
    (df_marker_expr['gene'].isin(key_display_markers))
]

sample_pivot = sample_data.pivot(index='cluster', columns='gene', values='pct_positive')
sample_pivot = sample_pivot[key_display_markers]  # Reorder columns
print(sample_pivot.round(1))
print(f"\nNote: Values shown are percent positive (% cells expressing > 0)")

print(f"\n{'='*80}")
print("OUTPUT FILES GENERATED:")
print(f"{'='*80}")
print(f"  1. tcell_marker_expression_per_cluster_long.csv")
print(f"     - Long format: cluster, gene, mean_expr, pct_positive, n_cells")
print(f"  2. tcell_marker_mean_expression_wide.csv")
print(f"     - Wide format: rows=clusters, columns=genes, values=mean_expr")
print(f"  3. tcell_marker_pct_positive_wide.csv")
print(f"     - Wide format: rows=clusters, columns=genes, values=pct_positive")
print(f"  4. tcell_marker_category_summary.csv")
print(f"     - Category-averaged: Pan_T_mean, CD4_T_mean, etc.")
print(f"{'='*80}\n")

In [ ]:
# ============================================================================
# VISUALIZATION: T/NK CELL ANNOTATIONS (SIMPLIFIED VERSION)
# ============================================================================

print("\n" + "="*80)
print("VISUALIZATION: T/NK CELL ANNOTATIONS - SIMPLIFIED NAMING")
print("="*80)

import matplotlib.pyplot as plt
import seaborn as sns

# ===== 1. Add annotation columns to AnnData =====

print("\n1. Adding annotation columns to AnnData...")

# Load SIMPLIFIED annotation table
annotation_file = '/home/h2048/data/R/0107/t_bbknn_filtered/tcell_3level_annotation_SIMPLIFIED.csv'
df_anno = pd.read_csv(annotation_file)

# Create mapping dictionaries using simplified column names
level1_map = dict(zip(df_anno['cluster'].astype(str), df_anno['Level_1']))
level2_map = dict(zip(df_anno['cluster'].astype(str), df_anno['Level_2']))
level3_map = dict(zip(df_anno['cluster'].astype(str), df_anno['Level_3']))
final_map = dict(zip(df_anno['cluster'].astype(str), df_anno['Final_Merged_ID']))
qc_map = dict(zip(df_anno['cluster'].astype(str), df_anno['QC_Flag']))
confidence_map = dict(zip(df_anno['cluster'].astype(str), df_anno['Confidence']))

# Apply to AnnData
adata.obs['Level_1'] = adata.obs['leiden'].map(level1_map)
adata.obs['Level_2'] = adata.obs['leiden'].map(level2_map)
adata.obs['Level_3'] = adata.obs['leiden'].map(level3_map)
adata.obs['cell_type_final'] = adata.obs['leiden'].map(final_map)
adata.obs['QC_flag'] = adata.obs['leiden'].map(qc_map)
adata.obs['annotation_confidence'] = adata.obs['leiden'].map(confidence_map)

print(f"✓ Added annotation columns:")
print(f"  - Level_1: {adata.obs['Level_1'].nunique()} unique types (NK, CD4+, CD8+, etc.)")
print(f"  - Level_2: {adata.obs['Level_2'].nunique()} unique states (CD4+ Trm, CD8+ Tem, etc.)")
print(f"  - Level_3: {adata.obs['Level_3'].nunique()} unique details")
print(f"  - cell_type_final: {adata.obs['cell_type_final'].nunique()} merged groups")

# ===== 2. Remove low quality cells =====

print("\n2. Filtering out low quality cells (C30, C36 only)...")

clusters_to_remove = ['30', '36']  # Only C30 and C36, C13 is kept as CD8+ Trm
n_cells_before = adata.n_obs
mask_keep = ~adata.obs['leiden'].isin(clusters_to_remove)
adata_clean = adata[mask_keep].copy()
n_cells_after = adata_clean.n_obs

print(f"  Cells before: {n_cells_before:,}")
print(f"  Cells after: {n_cells_after:,}")
print(f"  Removed: {n_cells_before - n_cells_after:,} cells (C30, C36)")
print(f"  ✓ C13 kept as CD8+ Trm (MT-high)")
print(f"  ✓ C6 kept as CD8+ Tem (stressed)")

# ===== 3. Define marker genes for visualization =====

print("\n3. Defining marker genes for visualization...")

# Core lineage markers
LINEAGE_MARKERS = ['CD3D', 'CD3E', 'CD4', 'CD8A', 'CD8B', 'GNLY', 'NKG7', 'FCGR3A']

# Functional state markers
FUNCTIONAL_MARKERS = [
    # Naive/Memory
    'CCR7', 'SELL', 'IL7R', 'TCF7',
    # Effector/Cytotoxic
    'GZMB', 'GZMH', 'PRF1', 'IFNG',
    # Tissue-resident
    'ITGAE', 'CXCR6', 'ZG16B', 'BPIFA1',
    # Treg
    'FOXP3', 'IL2RA', 'IKZF2',
    # Tfh
    'CXCR5', 'BCL6', 'ICOS', 'PDCD1',
    # Proliferation
    'MKI67', 'TOP2A',
    # Activation
    'FOS', 'JUN', 'XCL1', 'XCL2',
    # Unconventional
    'TRGV9', 'KLRB1'
]

# QC markers
QC_MARKERS = ['SCGB1A1', 'SCGB3A1', 'S100A8', 'S100A9', 'HSPA1B', 'HSPD1']

# All markers
ALL_MARKERS = LINEAGE_MARKERS + FUNCTIONAL_MARKERS + QC_MARKERS

# Filter to available genes
available_markers = [m for m in ALL_MARKERS if m in adata_clean.var_names]
print(f"  Total markers defined: {len(ALL_MARKERS)}")
print(f"  Available in dataset: {len(available_markers)}")

# ===== 4. UMAP: Annotations Overview =====

print("\n4. Generating UMAP plots...")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Plot 1: Original leiden clusters
sc.pl.umap(adata_clean, color='leiden', ax=axes[0, 0], show=False, 
           title='Original Leiden Clusters', legend_loc='on data', 
           legend_fontsize=8, size=UMAP_SIZE)

# Plot 2: Level 1 (Major cell type - simplified)
sc.pl.umap(adata_clean, color='Level_1', ax=axes[0, 1], show=False,
           title='Level 1: Major Cell Type', legend_loc='right margin',
           size=UMAP_SIZE, legend_fontsize=10)

# Plot 3: Level 2 (Cell type + State)
sc.pl.umap(adata_clean, color='Level_2', ax=axes[0, 2], show=False,
           title='Level 2: Cell Type + State', legend_loc='right margin',
           legend_fontsize=6, size=UMAP_SIZE)

# Plot 4: Final merged groups
sc.pl.umap(adata_clean, color='cell_type_final', ax=axes[1, 0], show=False,
           title='Final Merged Annotations', legend_loc='right margin',
           legend_fontsize=7, size=UMAP_SIZE)

# Plot 5: QC flags
sc.pl.umap(adata_clean, color='QC_flag', ax=axes[1, 1], show=False,
           title='QC Flags', legend_loc='right margin',
           size=UMAP_SIZE, legend_fontsize=8)

# Plot 6: Confidence
sc.pl.umap(adata_clean, color='annotation_confidence', ax=axes[1, 2], show=False,
           title='Annotation Confidence', legend_loc='right margin',
           size=UMAP_SIZE)

plt.tight_layout()
plt.savefig(fig_dir / f'08_annotation_overview_umap.{FIGURE_FORMAT}', 
            dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f"  ✓ Saved: 08_annotation_overview_umap.{FIGURE_FORMAT}")

# ===== 5. UMAP: Key Lineage Markers =====

print("\n5. Generating lineage marker UMAPs...")

lineage_markers_plot = ['CD3D', 'CD4', 'CD8A', 'GNLY', 'NKG7', 'FCGR3A']
lineage_markers_plot = [m for m in lineage_markers_plot if m in adata_clean.var_names]

sc.pl.umap(adata_clean, color=lineage_markers_plot, 
           ncols=3, use_raw=True, cmap='RdBu_r',
           vmin='p1', vmax='p99', size=UMAP_SIZE,
           save=f'_09_lineage_markers.{FIGURE_FORMAT}')
print(f"  ✓ Saved: umap_09_lineage_markers.{FIGURE_FORMAT}")

# ===== 6. Dotplot: Comprehensive Marker Expression =====

print("\n6. Generating comprehensive dotplot...")

# Define marker categories for dotplot
marker_dict_plot = {
    'Pan-T': ['CD3D', 'CD3E'],
    'Lineage': ['CD4', 'CD8A', 'CD8B'],
    'NK': ['GNLY', 'NKG7', 'FCGR3A', 'KLRD1'],
    'Naive': ['CCR7', 'SELL', 'IL7R'],
    'Memory': ['GZMK', 'CD69'],
    'Effector': ['GZMB', 'GZMH', 'PRF1', 'IFNG'],
    'Tissue-resident': ['ITGAE', 'CXCR6', 'ZG16B'],
    'Treg': ['FOXP3', 'IL2RA'],
    'Tfh': ['CXCR5', 'BCL6', 'ICOS'],
    'Proliferation': ['MKI67', 'TOP2A'],
    'Activation': ['FOS', 'JUN'],
    'QC': ['SCGB1A1', 'S100A8', 'HSPA1B']
}

# Flatten and filter
dotplot_markers = []
for category, genes in marker_dict_plot.items():
    dotplot_markers.extend([g for g in genes if g in adata_clean.var_names])

# Create dotplot grouped by Level 1
sc.pl.dotplot(
    adata_clean,
    var_names=dotplot_markers,
    groupby='Level_1',
    use_raw=True,
    figsize=(16, 5),
    dendrogram=True,
    swap_axes=False,
    save=f'_10_markers_by_L1.{FIGURE_FORMAT}'
)
print(f"  ✓ Saved: dotplot_10_markers_by_L1.{FIGURE_FORMAT}")

# Create dotplot grouped by Level 2
sc.pl.dotplot(
    adata_clean,
    var_names=dotplot_markers,
    groupby='Level_2',
    use_raw=True,
    figsize=(18, 10),
    dendrogram=True,
    save=f'_11_markers_by_L2.{FIGURE_FORMAT}'
)
print(f"  ✓ Saved: dotplot_11_markers_by_L2.{FIGURE_FORMAT}")

# ===== 7. Heatmap: Top markers per final group =====

print("\n7. Generating marker heatmap...")

# For final merged groups
try:
    sc.pl.rank_genes_groups_heatmap(
        adata_clean,
        n_genes=5,
        key='rank_genes_wilcox',
        use_raw=True,
        show=False,
        cmap='RdBu_r',
        figsize=(16, 14),
        vmin=-3,
        vmax=3,
        dendrogram=True,
        swap_axes=True
    )
    plt.savefig(fig_dir / f'12_top_markers_heatmap.{FIGURE_FORMAT}', 
                dpi=FIGURE_DPI, bbox_inches='tight')
    plt.show()
    print(f"  ✓ Saved: 12_top_markers_heatmap.{FIGURE_FORMAT}")
except Exception as e:
    print(f"  ⚠️ Heatmap skipped: {e}")
    print(f"     Run differential expression first if needed")

# ===== 8. Violin plots: Key markers by Level 1 =====

print("\n8. Generating violin plots...")

key_markers_violin = ['CD3D', 'CD4', 'CD8A', 'GNLY', 'FOXP3', 'MKI67']
key_markers_violin = [m for m in key_markers_violin if m in adata_clean.var_names]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, marker in enumerate(key_markers_violin):
    sc.pl.violin(
        adata_clean,
        keys=marker,
        groupby='Level_1',
        use_raw=True,
        ax=axes[idx],
        show=False,
        rotation=45
    )
    axes[idx].set_title(marker, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(fig_dir / f'13_key_markers_violin.{FIGURE_FORMAT}', 
            dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f"  ✓ Saved: 13_key_markers_violin.{FIGURE_FORMAT}")

# ===== 9. Stacked barplot: Cell type composition =====

print("\n9. Generating composition plots...")

# Cell type distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By Level 1
level1_counts = adata_clean.obs['Level_1'].value_counts()
axes[0].barh(level1_counts.index, level1_counts.values, color='steelblue')
axes[0].set_xlabel('Number of Cells', fontsize=12)
axes[0].set_title('Cell Type Distribution (Level 1)', fontsize=14, fontweight='bold')
for i, (ct, count) in enumerate(level1_counts.items()):
    axes[0].text(count + 200, i, f'{count:,}', va='center', fontsize=11)

# By Final merged
final_counts = adata_clean.obs['cell_type_final'].value_counts()
final_counts = final_counts[final_counts.index != 'REMOVE']
axes[1].barh(range(len(final_counts)), final_counts.values, color='coral')
axes[1].set_xlabel('Number of Cells', fontsize=12)
axes[1].set_title('Cell Type Distribution (Final Merged)', fontsize=14, fontweight='bold')
axes[1].set_yticks(range(len(final_counts)))
axes[1].set_yticklabels(final_counts.index, fontsize=8)
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)

# Add counts
for i, count in enumerate(final_counts.values):
    axes[1].text(count + 60, i, f'{count:,}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(fig_dir / f'14_celltype_composition.{FIGURE_FORMAT}', 
            dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f"  ✓ Saved: 14_celltype_composition.{FIGURE_FORMAT}")

# ===== 10. QC marker expression by cell type =====

print("\n10. Generating QC marker plots...")

qc_markers_plot = ['SCGB1A1', 'SCGB3A1', 'S100A8', 'S100A9', 'HSPA1B']
qc_markers_plot = [m for m in qc_markers_plot if m in adata_clean.var_names]

if len(qc_markers_plot) > 0:
    sc.pl.dotplot(
        adata_clean,
        var_names=qc_markers_plot,
        groupby='Level_1',
        use_raw=True,
        figsize=(8, 5),
        title='QC Markers by Cell Type',
        save=f'_15_qc_markers.{FIGURE_FORMAT}'
    )
    print(f"  ✓ Saved: dotplot_15_qc_markers.{FIGURE_FORMAT}")

# ===== 11. Summary statistics =====

print("\n" + "="*80)
print("ANNOTATION SUMMARY")
print("="*80)

print(f"\nTotal cells after QC: {adata_clean.n_obs:,}")
print(f"Cells removed (C30, C36 only): {n_cells_before - n_cells_after:,}")
print(f"✓ C13 (1,478 cells) kept as CD8+ Trm (MT-high)")
print(f"✓ C6 (1,829 cells) kept as CD8+ Tem (stressed)")

print(f"\nCell type distribution (Level 1):")
for ct, count in level1_counts.items():
    pct = count / adata_clean.n_obs * 100
    print(f"  {ct}: {count:,} cells ({pct:.1f}%)")

print(f"\nQC flags:")
qc_counts = adata_clean.obs['QC_flag'].value_counts()
for flag, count in qc_counts.items():
    pct = count / adata_clean.n_obs * 100
    print(f"  {flag}: {count:,} cells ({pct:.1f}%)")

print(f"\nAnnotation confidence:")
conf_counts = adata_clean.obs['annotation_confidence'].value_counts()
for conf, count in conf_counts.items():
    pct = count / adata_clean.n_obs * 100
    print(f"  {conf}: {count:,} cells ({pct:.1f}%)")

# ===== 12. Save annotated data =====

print("\n" + "="*80)
print("SAVING ANNOTATED DATA")
print("="*80)

output_h5ad = output_dir / "tcell_bbknn_annotated_simplified.h5ad"

print(f"\nSaving annotated AnnData to: {output_h5ad}")
print(f"  Contents:")
print(f"    - Cells: {adata_clean.n_obs:,}")
print(f"    - Genes: {adata_clean.n_vars:,}")
print(f"    - Full genes in .raw: {adata_clean.raw.n_vars:,}")
print(f"    - Annotation columns:")
print(f"      * Level_1 (NK, CD4+, CD8+, γδ T, MAIT)")
print(f"      * Level_2 (CD4+ Trm, CD8+ Tem, NK Activated, etc.)")
print(f"      * Level_3 (CD4+ Trm (activated), etc.)")
print(f"      * cell_type_final (23 merged groups)")
print(f"      * QC_flag")
print(f"      * annotation_confidence")

adata_clean.write_h5ad(output_h5ad, compression='gzip')
print(f"\n✓ Data saved successfully")

print("\n" + "="*80)
print("VISUALIZATION COMPLETE")
print("="*80)
print(f"\nGenerated figures:")
print(f"  1. 08_annotation_overview_umap.{FIGURE_FORMAT}")
print(f"  2. umap_09_lineage_markers.{FIGURE_FORMAT}")
print(f"  3. dotplot_10_markers_by_L1.{FIGURE_FORMAT}")
print(f"  4. dotplot_11_markers_by_L2.{FIGURE_FORMAT}")
print(f"  5. 12_top_markers_heatmap.{FIGURE_FORMAT}")
print(f"  6. 13_key_markers_violin.{FIGURE_FORMAT}")
print(f"  7. 14_celltype_composition.{FIGURE_FORMAT}")
print(f"  8. dotplot_15_qc_markers.{FIGURE_FORMAT}")
print("="*80)

## Step 12: Save Results

In [ ]:
# ============================================================================
# STEP 12: SAVE PROCESSED DATA
# ============================================================================

print("\n" + "="*80)
print("STEP 12: SAVING PROCESSED DATA")
print("="*80)

output_h5ad = output_dir / "tcell_bbknn_processed.h5ad"

print(f"\nSaving AnnData object...")
print(f"  Output: {output_h5ad}")
print(f"  Contents:")
print(f"    - Cells: {adata.n_obs:,}")
print(f"    - HVGs: {adata.n_vars:,}")
print(f"    - Full genes in .raw: {adata.raw.n_vars:,}")
print(f"    - BBKNN neighbors: ✓")
print(f"    - UMAP: ✓")
print(f"    - Leiden clusters: ✓")
print(f"    - Wilcoxon DE: ✓")

adata.write_h5ad(output_h5ad, compression='gzip')
print(f"\n✓ Data saved successfully")

## Summary

In [ ]:
# ============================================================================
# ANALYSIS SUMMARY
# ============================================================================

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)

print(f"\nOutput directory: {output_dir}")

print(f"\nGenerated files:")
print(f"  ├── tcell_bbknn_processed.h5ad")
print(f"  ├── tcell_markers_wilcox_all.csv")
print(f"  ├── tcell_markers_wilcox_top10.csv")
print(f"  └── figures/")
print(f"      ├── 01_umap_clusters.{FIGURE_FORMAT}")
print(f"      ├── 02_umap_batch.{FIGURE_FORMAT}")
print(f"      ├── 03_umap_combined.{FIGURE_FORMAT}")
print(f"      ├── 04_tcell_markers_umap.{FIGURE_FORMAT}")
print(f"      ├── 05_tcell_markers_dotplot.{FIGURE_FORMAT}")
print(f"      ├── 06_marker_heatmap.{FIGURE_FORMAT}")
print(f"      └── 07_marker_dotplot.{FIGURE_FORMAT}")

if len(markers_df) > 0:
    print(f"\nMarker summary by cluster:")
    for cluster in sorted(markers_df['cluster'].unique()):
        cluster_markers = markers_df[markers_df['cluster'] == cluster]
        top3 = cluster_markers.head(3)['names'].tolist()
        print(f"  Cluster {cluster}: {len(cluster_markers)} markers")
        print(f"    Top 3: {', '.join(top3)}")

print("\n" + "="*80)
print("Next steps:")
print("  1. Review canonical T cell markers (CD3D, CD4, CD8A, etc.)")
print("  2. Identify potential T cell subtypes:")
print("     - Naive T (CCR7+, SELL+, IL7R+)")
print("     - Memory T (GZMK+, CD69+)")
print("     - Effector T (GZMB+, PRF1+)")
print("     - Treg (FOXP3+, IL2RA+)")
print("     - NK (GNLY+, NKG7+)")
print("  3. Consider trajectory analysis for differentiation dynamics")
print("  4. Compare CD4+ vs CD8+ subsets if both present")
print("="*80)